# Задача 7: Оценка погрешности по правилу Рунге
---

**Задача:** Проверить оценку погрешности Рунге для интегрирования функции на отрезке [0, 1] методом трапеций при значениях числа отрезков разбиения 4 и 8, 8 и 16, 16 и 32.

### Теория

**Правило Рунге** позволяет оценить погрешность численного метода, сравнивая результаты с разными шагами.

Для метода порядка точности $p$ при уменьшении шага в 2 раза:

$$|I - I_h| \approx \frac{|I_h - I_{h/2}|}{2^p - 1}$$

где:
- $I$ — точное значение интеграла
- $I_h$ — приближение с шагом $h$
- $I_{h/2}$ — приближение с шагом $h/2$

**Метод трапеций** имеет порядок точности $p = 2$, поэтому:

$$|I - I_h| \approx \frac{|I_h - I_{h/2}|}{3}$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

### Функция для интегрирования

Используем $f(x) = e^x$ на отрезке $[0, 1]$.

Точное значение: $\int_0^1 e^x dx = e - 1 \approx 1.718281828$

In [ ]:
def f(x):
    """Функция для интегрирования: f(x) = e^x"""
    return np.exp(x)

# Точное значение интеграла
I_exact = np.e - 1
print(f"Точное значение интеграла: {I_exact:.10f}")

### Метод трапеций

In [ ]:
def trapezoidal(f, a, b, n):
    """
    Вычисление интеграла методом трапеций.
    
    Параметры:
    f : функция для интегрирования
    a, b : границы интегрирования
    n : число отрезков разбиения
    
    Возвращает:
    Приближенное значение интеграла
    """
    h = (b - a) / n
    x = np.linspace(a, b, n + 1)
    y = f(x)
    
    # Формула трапеций: I ≈ h * (y0/2 + y1 + y2 + ... + y_{n-1} + yn/2)
    integral = h * (y[0]/2 + np.sum(y[1:-1]) + y[-1]/2)
    
    return integral

### Вычисление интегралов с разным числом разбиений

In [ ]:
# Границы интегрирования
a, b = 0, 1

# Числа отрезков разбиения
n_values = [4, 8, 16, 32]

# Вычисляем интегралы
results = {}
for n in n_values:
    I_n = trapezoidal(f, a, b, n)
    error_actual = abs(I_exact - I_n)
    results[n] = {'value': I_n, 'error': error_actual}
    print(f"n = {n:2d}: I = {I_n:.10f}, погрешность = {error_actual:.2e}")

### Проверка оценки погрешности по правилу Рунге

In [ ]:
def runge_error_estimate(I_h, I_h2, p=2):
    """
    Оценка погрешности по правилу Рунге.
    
    Параметры:
    I_h : значение интеграла с шагом h
    I_h2 : значение интеграла с шагом h/2
    p : порядок точности метода (для трапеций p=2)
    
    Возвращает:
    Оценку погрешности для I_h
    """
    return abs(I_h - I_h2) / (2**p - 1)

print("="*80)
print("Проверка оценки погрешности по правилу Рунге")
print("="*80)
print(f"{'Пара (n1, n2)':<20} {'Оценка Рунге':<20} {'Фактическая':<20} {'Отношение':<15}")
print("-"*80)

pairs = [(4, 8), (8, 16), (16, 32)]

for n1, n2 in pairs:
    I_n1 = results[n1]['value']
    I_n2 = results[n2]['value']
    
    # Оценка погрешности по Рунге для I_n1
    error_runge = runge_error_estimate(I_n1, I_n2, p=2)
    
    # Фактическая погрешность для I_n1
    error_actual = results[n1]['error']
    
    # Отношение оценки к фактической погрешности
    ratio = error_runge / error_actual if error_actual > 0 else 0
    
    print(f"({n1:2d}, {n2:2d})            {error_runge:.10e}    {error_actual:.10e}    {ratio:.4f}")

print("-"*80)
print("\nВывод: Оценка по правилу Рунге близка к фактической погрешности.")
print("Отношение должно быть близко к 1, что подтверждает корректность оценки.")

### Визуализация

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# График 1: Сходимость интеграла
n_plot = n_values
I_plot = [results[n]['value'] for n in n_values]
errors_plot = [results[n]['error'] for n in n_values]

ax1.plot(n_plot, I_plot, 'o-', label='Приближенное значение', linewidth=2, markersize=8)
ax1.axhline(y=I_exact, color='r', linestyle='--', label=f'Точное значение = {I_exact:.6f}')
ax1.set_xlabel('Число отрезков n', fontsize=12)
ax1.set_ylabel('Значение интеграла', fontsize=12)
ax1.set_title('Сходимость метода трапеций', fontsize=14)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# График 2: Погрешность в логарифмическом масштабе
ax2.loglog(n_plot, errors_plot, 'o-', label='Фактическая погрешность', linewidth=2, markersize=8)

# Добавляем оценки Рунге
runge_estimates = []
runge_n = []
for n1, n2 in pairs:
    I_n1 = results[n1]['value']
    I_n2 = results[n2]['value']
    error_runge = runge_error_estimate(I_n1, I_n2, p=2)
    runge_estimates.append(error_runge)
    runge_n.append(n1)

ax2.loglog(runge_n, runge_estimates, 's--', label='Оценка Рунге', linewidth=2, markersize=8)

# Теоретическая линия O(h^2) = O(1/n^2)
n_theory = np.array(n_plot)
error_theory = errors_plot[0] * (n_plot[0] / n_theory)**2
ax2.loglog(n_theory, error_theory, ':', color='gray', label='O(1/n²)', linewidth=2)

ax2.set_xlabel('Число отрезков n', fontsize=12)
ax2.set_ylabel('Погрешность', fontsize=12)
ax2.set_title('Погрешность метода трапеций', fontsize=14)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()